In [1]:
import json
import pandas as pd

with open("data/countries.json", "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df_raw = pd.json_normalize(raw_data)

display(df_raw.head())

,tld,cca2,ccn3,cca3,cioc,independent,status,unMember,unRegionalGroup,capital,...,currencies.ZWG.name,currencies.ZWG.symbol,languages.bwg,languages.kck,languages.khi,languages.ndc,languages.nde,languages.sna,languages.toi,languages.zib
0,[.aw],AW,533,ABW,ARU,False,officially-assigned,False,,[Oranjestad],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[.af],AF,004,AFG,AFG,True,officially-assigned,True,Asia and the Pacific Group,[Kabul],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[.ao],AO,024,AGO,ANG,True,officially-assigned,True,African Group,[Luanda],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[.ai],AI,660,AIA,,False,officially-assigned,False,,[The Valley],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[.ax],AX,248,ALA,,False,officially-assigned,False,,[Mariehamn],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
print("Shape:", df_raw.shape)

df_raw.info()

Shape: (250, 853)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Columns: 853 entries, tld to languages.zib
dtypes: bool(2), float64(1), object(850)
memory usage: 1.6+ MB


In [3]:
missing = df_raw.isnull().sum()

display(missing.sort_values(ascending=False).head(15))

languages.zib               249
name.native.isl.common      249
currencies.ISK.symbol       249
languages.isl               249
name.native.heb.official    249
name.native.heb.common      249
languages.heb               249
name.native.jam.official    249
name.native.jam.common      249
currencies.JMD.name         249
currencies.JMD.symbol       249
languages.jam               249
name.native.nrf.official    249
name.native.nrf.common      249
currencies.JEP.name         249
dtype: int64

In [4]:
df = df_raw[
    [
        "name.common",
        "cca3",
        "capital",
        "flag",
        "region",
        "independent"
    ]
].copy()

df.columns = [
    "country",
    "code",
    "capital",
    "flag",
    "region",
    "independent"
]

display(df.head())

,country,code,capital,flag,region,independent
0,Aruba,ABW,[Oranjestad],🇦🇼,Americas,False
1,Afghanistan,AFG,[Kabul],🇦🇫,Asia,True
2,Angola,AGO,[Luanda],🇦🇴,Africa,True
3,Anguilla,AIA,[The Valley],🇦🇮,Americas,False
4,Åland Islands,ALA,[Mariehamn],🇦🇽,Europe,False


In [5]:
df = df[df["independent"] == True].copy()

In [6]:
def get_capital(capitals):
    if isinstance(capitals, list) and len(capitals) == 1:
        return capitals[0]
    return None


df["capital"] = df["capital"].apply(get_capital)

In [7]:
df = df.dropna(
    subset=["country", "code", "capital", "flag", "region"]
)

In [8]:
columns = ["country", "code", "capital", "flag", "region"]

for column in columns:
    df[column] = df[column].str.strip()
    df = df[df[column] != ""]

In [9]:
df = df.drop_duplicates(subset="code")
df = df.drop_duplicates(subset="capital")

df = df.reset_index(drop=True)

In [10]:
print("raw records:", len(df_raw))
print("clean records:", len(df))

display(df.head(10))

raw records: 250
clean records: 193


,country,code,capital,flag,region,independent
0,Afghanistan,AFG,Kabul,🇦🇫,Asia,True
1,Angola,AGO,Luanda,🇦🇴,Africa,True
2,Albania,ALB,Tirana,🇦🇱,Europe,True
3,Andorra,AND,Andorra la Vella,🇦🇩,Europe,True
4,United Arab Emirates,ARE,Abu Dhabi,🇦🇪,Asia,True
5,Argentina,ARG,Buenos Aires,🇦🇷,Americas,True
6,Armenia,ARM,Yerevan,🇦🇲,Asia,True
7,Antigua and Barbuda,ATG,Saint John's,🇦🇬,Americas,True
8,Australia,AUS,Canberra,🇦🇺,Oceania,True
9,Austria,AUT,Vienna,🇦🇹,Europe,True


In [11]:
print(df.isnull().sum())

country        0
code           0
capital        0
flag           0
region         0
independent    0
dtype: int64


In [12]:
print("duplicate countries:", df["code"].duplicated().sum())
print("duplicate capitals:", df["capital"].duplicated().sum())

duplicate countries: 0
duplicate capitals: 0


In [13]:
df.to_csv("data/countries_clean.csv", index=False)

In [15]:
from atw.data_loader import load_countries

countries = load_countries("data/countries.json")

display(countries.head())

,country,code,capital,flag,region
0,Afghanistan,AFG,Kabul,🇦🇫,Asia
1,Angola,AGO,Luanda,🇦🇴,Africa
2,Albania,ALB,Tirana,🇦🇱,Europe
3,Andorra,AND,Andorra la Vella,🇦🇩,Europe
4,United Arab Emirates,ARE,Abu Dhabi,🇦🇪,Asia


In [16]:
print("number of countries:", len(countries))
print("missing values:", countries.isnull().sum().sum())
print("duplicate capitals:", countries["capital"].duplicated().sum())

number of countries: 193
missing values: 0
duplicate capitals: 0


In [17]:
countries.to_csv("data/countries_clean.csv", index=False)

In [18]:
from atw.data_loader import load_countries
from atw.questions import generate_capital_question

countries = load_countries("data/countries.json")

countries_list = countries.to_dict("records")

question = generate_capital_question(countries_list)

print(question["question"])

for i, option in enumerate(question["options"], 1):
    print(f"{i}. {option}")

What is the capital of Eritrea?
1. Tirana
2. Asmara
3. Ngerulmud
4. Warsaw


In [19]:
print(question["question"])
print(question["options"])
print("Correct answer:", question["correct_answer"])

What is the capital of Eritrea?
['Tirana', 'Asmara', 'Ngerulmud', 'Warsaw']
Correct answer: Asmara


In [1]:
from atw.data_loader import load_countries
from atw.questions import generate_quiz

countries = load_countries("data/countries.json")
countries_list = countries.to_dict("records")

quiz = generate_quiz(countries_list, 10)

print("Number of questions:", len(quiz))

Number of questions: 10


In [2]:
for i, question in enumerate(quiz, 1):
    print(f"\nQuestion {i}")
    print(question["question"])

    for number, option in enumerate(question["options"], 1):
        print(f"{number}. {option}")


Question 1
What is the capital of Canada?
1. Prague
2. Ottawa
3. Bishkek
4. Seoul

Question 2
What is the capital of Eswatini?
1. Lobamba
2. Sofia
3. Port Vila
4. Banjul

Question 3
What is the capital of Monaco?
1. Bissau
2. Washington D.C.
3. Asunción
4. Monaco

Question 4
What is the capital of Saudi Arabia?
1. Yerevan
2. Zagreb
3. Port Moresby
4. Riyadh

Question 5
What is the capital of Bhutan?
1. Luanda
2. Manama
3. Thimphu
4. Bandar Seri Begawan

Question 6
What is the capital of Cape Verde?
1. Roseau
2. Libreville
3. Malé
4. Praia

Question 7
What is the capital of Central African Republic?
1. Funafuti
2. Riyadh
3. Bangui
4. Washington D.C.

Question 8
What is the capital of Latvia?
1. São Tomé
2. Riga
3. Rabat
4. Tegucigalpa

Question 9
What is the capital of India?
1. Kuwait City
2. New Delhi
3. South Tarawa
4. City of San Marino

Question 10
What is the capital of Oman?
1. Mexico City
2. Brazzaville
3. Sofia
4. Muscat


In [3]:
correct_countries = [
    question["correct_country"] for question in quiz
]

print(correct_countries)
print("Unique countries:", len(set(correct_countries)))

['Canada', 'Eswatini', 'Monaco', 'Saudi Arabia', 'Bhutan', 'Cape Verde', 'Central African Republic', 'Latvia', 'India', 'Oman']
Unique countries: 10


In [4]:
from atw.population import load_population

population = load_population(2025)

display(population.head())

print("Records:", len(population))

,code,population,year
0,AFE,788844284,2025
1,AFW,532809933,2025
2,ARB,503356133,2025
3,CSS,4556558,2025
4,CEB,99814596,2025


Records: 261


In [5]:
population.to_csv(
    "data/population_2025.csv",
    index=False
)

In [6]:
from atw.data_loader import load_countries
from atw.population import merge_population

countries = load_countries("data/countries.json")

combined = merge_population(countries, population)

display(combined.head())

,country,code,capital,flag,region,lat,lon,population,year
0,Afghanistan,AFG,Kabul,🇦🇫,Asia,33.0,65.0,43844111,2025
1,Angola,AGO,Luanda,🇦🇴,Africa,-12.5,18.5,39040039,2025
2,Albania,ALB,Tirana,🇦🇱,Europe,41.0,20.0,2349580,2025
3,Andorra,AND,Andorra la Vella,🇦🇩,Europe,42.5,1.5,82904,2025
4,United Arab Emirates,ARE,Abu Dhabi,🇦🇪,Asia,24.0,54.0,11513149,2025


In [9]:
print("Countries before merge:", len(countries))
print("Countries after merge:", len(combined))

print("Missing values:")
print(combined.isnull().sum())

assert combined["population"].gt(0).all()
assert combined["year"].eq(2025).all()
assert combined["code"].is_unique

Countries before merge: 193
Countries after merge: 192
Missing values:
country       0
code          0
capital       0
flag          0
region        0
lat           0
lon           0
population    0
year          0
dtype: int64


In [10]:
combined.to_csv(
    "data/countries_population.csv",
    index=False
)